# Few-Shot PCB Defect Segmentation Pipeline

Readable end-to-end notebook for the ECE4880J final project:
**Few-Shot PCB Defect Segmentation via Multi-Scale DINOv2 Anomaly Proposals and SAM2 Mask Refinement**.

The notebook is a presentation and reproduction layer. Core implementation stays in `src/`, runnable experiments stay in `scripts/`, and generated data/model/output artifacts stay outside Git.


## How To Use This Notebook

This notebook supports two modes:

- **Read/inspect mode:** leave the run flags as `False`. The notebook loads existing manifests, saved metrics, and qualitative artifacts from `outputs/` if they exist.
- **Reproduction mode:** set `RUN_EXPENSIVE = True` and, optionally, `RUN_SAM2 = True` to regenerate experiments from the same commands shown here.

Assumptions:

- VisA PCB manifests are under `data/manifests/`.
- Official VisA images are under `data/processed/VisA_pytorch/1cls`.
- SAM2 is optional and requires local ignored assets under `external/` and `weights/`.
- DeepPCB is treated as box-level localization or pseudo-mask visualization only, not true segmentation ground truth.


In [ ]:
from __future__ import annotations

import csv
import json
import os
import subprocess
import sys
from collections import Counter
from pathlib import Path

import numpy as np
from PIL import Image, ImageDraw

try:
    from IPython.display import Markdown, display
except ModuleNotFoundError:
    class Markdown(str):
        pass

    def display(obj):
        print(obj)

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from utils.config import load_yaml
from utils.visualize import safe_filename
from scripts.run_dinov2_baseline import select_rows

config = load_yaml(PROJECT_ROOT / "configs" / "default.yaml")
CATEGORIES = list(config["datasets"]["visa_pcb"]["categories"])

RUN_EXPENSIVE = False
RUN_SAM2 = False
FOLD_ID = 0
K_SHOT = int(config["few_shot"]["k"])
SEED = int(config["few_shot"]["seed"])
MULTISCALE_CROP_SIZES = "768"
MULTISCALE_OVERLAP = "0.25"
MAX_MASK_AREA_FRACTION = None

print(config["project"]["title"])
print(f"project root: {PROJECT_ROOT}")
print(f"categories: {CATEGORIES}")


In [ ]:
def read_csv_rows(path: str | Path) -> list[dict[str, str]]:
    path = Path(path)
    with path.open(newline="") as handle:
        return list(csv.DictReader(handle))


def read_json(path: str | Path) -> dict:
    return json.loads(Path(path).read_text())


def display_markdown_file(path: str | Path) -> None:
    path = Path(path)
    if path.exists():
        display(Markdown(path.read_text()))
    else:
        display(Markdown(f"Missing `{path.relative_to(PROJECT_ROOT)}`. Run the corresponding commands below."))


def markdown_table(headers: list[str], rows: list[list[object]]) -> str:
    lines = ["| " + " | ".join(headers) + " |"]
    lines.append("| " + " | ".join(["---"] * len(headers)) + " |")
    for row in rows:
        lines.append("| " + " | ".join(str(item) for item in row) + " |")
    return "\n".join(lines)


def run_repo_command(args: list[str], *, enabled: bool = False) -> None:
    command = [sys.executable] + args
    rel_command = " ".join(str(part) for part in command)
    display(Markdown(f"```bash\nPYTHONPATH=src {rel_command}\n```"))
    if not enabled:
        return
    env = os.environ.copy()
    env["PYTHONPATH"] = str(SRC_ROOT)
    subprocess.run(command, cwd=PROJECT_ROOT, env=env, check=True)


## 1. Dataset Manifests

The loaders normalize VisA and DeepPCB into manifest rows with paths and labels. VisA PCB folds are used for segmentation evaluation. DeepPCB is kept as a secondary PCB-specific localization dataset with boxes, not true segmentation masks.


In [ ]:
manifest_dir = PROJECT_ROOT / "data" / "manifests"
manifest_paths = {
    "visa_pcb_manifest": manifest_dir / "visa_pcb_manifest.csv",
    "visa_pcb_folds": manifest_dir / "visa_pcb_folds.csv",
    "deeppcb_manifest": manifest_dir / "deeppcb_manifest.csv",
    "deeppcb_folds": manifest_dir / "deeppcb_folds.csv",
}

summary_rows = []
for name, path in manifest_paths.items():
    if not path.exists():
        summary_rows.append([name, "missing", "-", "-"])
        continue
    rows = read_csv_rows(path)
    split_counts = dict(sorted(Counter(row.get("split", "") for row in rows).items()))
    category_counts = dict(sorted(Counter(row.get("category", "") for row in rows).items()))
    summary_rows.append([name, len(rows), split_counts, category_counts])

display(Markdown(markdown_table(["manifest", "rows", "splits", "categories"], summary_rows)))


## 2. Few-Shot Support Sampling

For each VisA PCB category, the model uses `k` normal images from the development fold as the support set. Query images come from the selected test fold and include both normal and anomalous samples.


In [ ]:
def overlay_mask(image: Image.Image, mask_path: str | Path | None, alpha: int = 110) -> Image.Image:
    image = image.convert("RGBA")
    if not mask_path:
        return image.convert("RGB")
    mask_path = Path(mask_path)
    if not mask_path.exists():
        return image.convert("RGB")
    mask = Image.open(mask_path).convert("L").resize(image.size, Image.Resampling.NEAREST)
    red = Image.new("RGBA", image.size, (255, 0, 0, 0))
    red.putalpha(mask.point(lambda value: alpha if value > 0 else 0))
    return Image.alpha_composite(image, red).convert("RGB")


def make_sample_grid(rows: list[dict[str, str]], *, show_masks: bool = False, thumb_size=(180, 140)) -> Image.Image | None:
    if not rows:
        return None
    label_height = 34
    cols = min(4, len(rows))
    rows_count = (len(rows) + cols - 1) // cols
    tile_w, tile_h = thumb_size[0], thumb_size[1] + label_height
    canvas = Image.new("RGB", (cols * tile_w, rows_count * tile_h), "white")
    draw = ImageDraw.Draw(canvas)
    for index, row in enumerate(rows):
        x = (index % cols) * tile_w
        y = (index // cols) * tile_h
        image = Image.open(row["image_path"]).convert("RGB")
        image.thumbnail(thumb_size, Image.Resampling.BILINEAR)
        if show_masks:
            image = overlay_mask(image, row.get("mask_path"), alpha=120)
        canvas.paste(image, (x, y))
        label = f"{row.get('sample_id', '')}\nlabel={row.get('label', '')}"
        draw.text((x + 4, y + thumb_size[1] + 2), label[:52], fill=(0, 0, 0))
    return canvas

visa_fold_path = manifest_paths["visa_pcb_folds"]
if visa_fold_path.exists():
    visa_rows = read_csv_rows(visa_fold_path)
    support_rows, query_rows = select_rows(
        rows=visa_rows,
        fold_id=FOLD_ID,
        category="pcb1",
        k=K_SHOT,
        query_fold_split="test",
        limit=8,
        seed=SEED,
    )
    display(Markdown(f"`pcb1`, fold `{FOLD_ID}`, k=`{K_SHOT}` support examples:"))
    display(make_sample_grid(support_rows[:K_SHOT]))
    display(Markdown("First interleaved query examples with ground-truth masks where available:"))
    display(make_sample_grid(query_rows[:8], show_masks=True))
else:
    display(Markdown("Missing VisA fold manifest. Run `scripts/create_manifests.py` first."))


## 3. DINOv2 Memory Bank and Heatmaps

The baseline script performs the normal-only few-shot pipeline:

1. sample `k` normal support images,
2. extract DINOv2 patch features,
3. build a normal memory bank,
4. score query patches by distance to the memory bank,
5. save heatmaps, overlays, and `scores.csv`.

The multi-scale variant adds local crop inference and fuses crop heatmaps back into the full image grid.


In [ ]:
for category in CATEGORIES:
    run_repo_command(
        [
            "scripts/run_dinov2_baseline.py",
            "--manifest", "data/manifests/visa_pcb_folds.csv",
            "--fold-id", str(FOLD_ID),
            "--category", category,
            "--k", str(K_SHOT),
            "--limit", "100000",
            "--query-fold-split", "test",
            "--feature-backbone", "dinov2_vits14",
            "--image-size", "518",
            "--patch-size", "14",
            "--device", "auto",
            "--seed", str(SEED),
            "--output-dir", f"outputs/dinov2_vits14_{category}_fold0_full",
        ],
        enabled=RUN_EXPENSIVE,
    )
    run_repo_command(
        [
            "scripts/run_dinov2_baseline.py",
            "--manifest", "data/manifests/visa_pcb_folds.csv",
            "--fold-id", str(FOLD_ID),
            "--category", category,
            "--k", str(K_SHOT),
            "--limit", "100000",
            "--query-fold-split", "test",
            "--feature-backbone", "dinov2_vits14",
            "--image-size", "518",
            "--patch-size", "14",
            "--device", "auto",
            "--seed", str(SEED),
            "--crop-sizes", MULTISCALE_CROP_SIZES,
            "--crop-overlap", MULTISCALE_OVERLAP,
            "--fusion", "max",
            "--output-dir", f"outputs/dinov2_vits14_{category}_fold0_ms768_o025_max_full",
        ],
        enabled=RUN_EXPENSIVE,
    )


## 4. Heatmap Evaluation

VisA PCB heatmaps are evaluated with image AUROC, pixel AUROC, best pixel F1, and best pixel IoU. The current fast evaluation uses a deterministic sample of up to 1,000,000 pixels by default; pass `--max-pixels 0` for exact full-resolution pixel metrics.


In [ ]:
for category in CATEGORIES:
    for output_dir in [
        f"outputs/dinov2_vits14_{category}_fold0_full",
        f"outputs/dinov2_vits14_{category}_fold0_ms768_o025_max_full",
    ]:
        run_repo_command(
            [
                "scripts/evaluate_heatmaps.py",
                "--scores-csv", f"{output_dir}/scores.csv",
                "--output-json", f"{output_dir}/metrics.json",
                "--max-pixels", "1000000",
                "--seed", str(SEED),
            ],
            enabled=RUN_EXPENSIVE,
        )

display_markdown_file(PROJECT_ROOT / "outputs" / "summary" / "stage2_fold0_single_vs_ms768.md")


## 5. SAM2 Prompting and Mask Refinement

The SAM2-only baseline uses fixed image-grid prompts without DINOv2 anomaly guidance. The DINOv2+SAM2 variants convert high-score connected components into one box plus one positive point per candidate region. SAM2 candidate masks are ranked by SAM2 confidence, anomaly strength, prompt containment, and mask area. `--max-mask-area-fraction` can reject oversized mask candidates during selection.


In [ ]:
def sam2_only_args(category: str, output_dir: str) -> list[str]:
    args = [
        "scripts/run_sam2_baseline.py",
        "--manifest", "data/manifests/visa_pcb_folds.csv",
        "--fold-id", str(FOLD_ID),
        "--category", category,
        "--limit", "100000",
        "--query-fold-split", "test",
        "--prompt-longest-side", "128",
        "--grid-size", "3",
        "--max-regions", "9",
        "--refiner", "sam2",
        "--sam2-checkpoint", "weights/sam2.1_hiera_tiny.pt",
        "--sam2-model-config", "configs/sam2.1/sam2.1_hiera_t.yaml",
        "--device", "auto",
        "--output-dir", output_dir,
    ]
    if MAX_MASK_AREA_FRACTION is not None:
        args += ["--max-mask-area-fraction", str(MAX_MASK_AREA_FRACTION)]
    return args


def sam2_refinement_args(category: str, source_dir: str, output_dir: str) -> list[str]:
    args = [
        "scripts/run_mask_refinement.py",
        "--scores-csv", f"{source_dir}/scores.csv",
        "--output-dir", output_dir,
        "--percentile", "95",
        "--min-area", "16",
        "--max-regions", "2",
        "--refiner", "sam2",
        "--sam2-checkpoint", "weights/sam2.1_hiera_tiny.pt",
        "--sam2-model-config", "configs/sam2.1/sam2.1_hiera_t.yaml",
        "--device", "auto",
    ]
    if MAX_MASK_AREA_FRACTION is not None:
        args += ["--max-mask-area-fraction", str(MAX_MASK_AREA_FRACTION)]
    return args

for category in CATEGORIES:
    sam2_only_dir = f"outputs/sam2_only_{category}_fold0_full"
    single_source = f"outputs/dinov2_vits14_{category}_fold0_full"
    multi_source = f"outputs/dinov2_vits14_{category}_fold0_ms768_o025_max_full"
    run_repo_command(
        sam2_only_args(category, sam2_only_dir),
        enabled=RUN_SAM2,
    )
    run_repo_command(
        sam2_refinement_args(category, single_source, f"{single_source}_sam2"),
        enabled=RUN_SAM2,
    )
    run_repo_command(
        sam2_refinement_args(category, multi_source, f"{multi_source}_sam2"),
        enabled=RUN_SAM2,
    )


In [ ]:
for category in CATEGORIES:
    sam2_only_dir = f"outputs/sam2_only_{category}_fold0_full"
    run_repo_command(
        [
            "scripts/evaluate_masks.py",
            "--mask-scores-csv", f"{sam2_only_dir}/mask_scores.csv",
            "--output-json", f"{sam2_only_dir}/mask_metrics.json",
            "--output-csv", f"{sam2_only_dir}/mask_metrics.csv",
        ],
        enabled=RUN_SAM2,
    )
    for source_dir, mask_dir in [
        (
            f"outputs/dinov2_vits14_{category}_fold0_full",
            f"outputs/dinov2_vits14_{category}_fold0_full_sam2",
        ),
        (
            f"outputs/dinov2_vits14_{category}_fold0_ms768_o025_max_full",
            f"outputs/dinov2_vits14_{category}_fold0_ms768_o025_max_full_sam2",
        ),
    ]:
        run_repo_command(
            [
                "scripts/evaluate_masks.py",
                "--mask-scores-csv", f"{mask_dir}/mask_scores.csv",
                "--source-scores-csv", f"{source_dir}/scores.csv",
                "--output-json", f"{mask_dir}/mask_metrics.json",
                "--output-csv", f"{mask_dir}/mask_metrics.csv",
            ],
            enabled=RUN_SAM2,
        )

stage4_metric_files = []
for category in CATEGORIES:
    stage4_metric_files.extend([
        f"outputs/dinov2_vits14_{category}_fold0_full/metrics.json",
        f"outputs/sam2_only_{category}_fold0_full/mask_metrics.json",
        f"outputs/dinov2_vits14_{category}_fold0_full_sam2/mask_metrics.json",
        f"outputs/dinov2_vits14_{category}_fold0_ms768_o025_max_full/metrics.json",
        f"outputs/dinov2_vits14_{category}_fold0_ms768_o025_max_full_sam2/mask_metrics.json",
    ])
run_repo_command(
    [
        "scripts/summarize_stage4.py",
        "--metrics-json",
        *stage4_metric_files,
        "--output-csv", "outputs/summary/stage4_fold0_multiscale_comparison.csv",
        "--output-md", "outputs/summary/stage4_fold0_multiscale_comparison.md",
    ],
    enabled=RUN_SAM2,
)

display_markdown_file(PROJECT_ROOT / "outputs" / "summary" / "stage4_fold0_multiscale_comparison.md")


## 6. Qualitative Visualization

The helper below loads one anomalous sample, its ground-truth mask, the multi-scale heatmap, and the SAM2 prediction. This is meant for success/failure inspection, not just nice figures.


In [ ]:
def heatmap_to_rgb(heatmap: np.ndarray) -> Image.Image:
    heatmap = heatmap.astype(np.float32, copy=False)
    minimum = float(heatmap.min())
    maximum = float(heatmap.max())
    if maximum > minimum:
        heatmap = (heatmap - minimum) / (maximum - minimum)
    else:
        heatmap = np.zeros_like(heatmap)
    red = (heatmap * 255).astype(np.uint8)
    green = (np.clip(1.0 - np.abs(heatmap - 0.75) * 2.0, 0.0, 1.0) * 220).astype(np.uint8)
    blue = ((1.0 - heatmap) * 80).astype(np.uint8)
    return Image.fromarray(np.stack([red, green, blue], axis=-1), mode="RGB")


def overlay_heatmap(image: Image.Image, heatmap: np.ndarray, alpha: float = 0.35) -> Image.Image:
    heatmap_rgb = heatmap_to_rgb(heatmap).resize(image.size, Image.Resampling.BILINEAR)
    return Image.blend(image.convert("RGB"), heatmap_rgb, alpha=alpha)


def make_qualitative_panel(category: str = "pcb1", sample_index: int = 0) -> Image.Image | None:
    score_path = PROJECT_ROOT / f"outputs/dinov2_vits14_{category}_fold0_ms768_o025_max_full/scores.csv"
    mask_score_path = PROJECT_ROOT / f"outputs/dinov2_vits14_{category}_fold0_ms768_o025_max_full_sam2/mask_scores.csv"
    if not score_path.exists() or not mask_score_path.exists():
        display(Markdown("Missing heatmap or SAM2 outputs. Run the experiment commands first."))
        return None

    score_rows = [row for row in read_csv_rows(score_path) if row.get("label") == "1"]
    mask_rows = {row["sample_id"]: row for row in read_csv_rows(mask_score_path)}
    if not score_rows:
        return None
    row = score_rows[min(sample_index, len(score_rows) - 1)]
    mask_row = mask_rows.get(row["sample_id"])
    if mask_row is None:
        display(Markdown(f"No SAM2 mask row for `{row['sample_id']}`."))
        return None

    image = Image.open(row["image_path"]).convert("RGB")
    heatmap = np.load(row["heatmap_path"])
    gt_overlay = overlay_mask(image, row.get("mask_path"), alpha=140)
    heatmap_overlay = overlay_heatmap(image, heatmap)
    pred_overlay = overlay_mask(image, mask_row.get("pred_mask_path"), alpha=140)

    panels = [image, gt_overlay, heatmap_overlay, pred_overlay]
    labels = ["image", "ground truth", "multi-scale heatmap", "SAM2 mask"]
    thumb_w, thumb_h = 260, 190
    label_h = 26
    canvas = Image.new("RGB", (len(panels) * thumb_w, thumb_h + label_h), "white")
    draw = ImageDraw.Draw(canvas)
    for index, (panel, label) in enumerate(zip(panels, labels)):
        panel = panel.copy()
        panel.thumbnail((thumb_w, thumb_h), Image.Resampling.BILINEAR)
        x = index * thumb_w
        canvas.paste(panel, (x, 0))
        draw.text((x + 4, thumb_h + 4), label, fill=(0, 0, 0))
    display(Markdown(f"Sample: `{row['sample_id']}`"))
    return canvas

panel = make_qualitative_panel("pcb1", sample_index=0)
if panel is not None:
    display(panel)


## 7. DeepPCB Boundary

DeepPCB is useful for PCB-specific localization and qualitative reference-based experiments. It has aligned template/test image pairs and bounding boxes, not pixel-accurate defect masks. In this project it should not be reported as true segmentation ground truth unless boxes are explicitly labeled as coarse pseudo-masks.


In [ ]:
deeppcb_path = manifest_paths["deeppcb_manifest"]
if deeppcb_path.exists():
    rows = read_csv_rows(deeppcb_path)
    class_counts = Counter(row.get("category", "") for row in rows)
    split_counts = Counter(row.get("split", "") for row in rows)
    display(Markdown(markdown_table(
        ["DeepPCB rows", "splits", "defect categories"],
        [[len(rows), dict(sorted(split_counts.items())), dict(sorted(class_counts.items()))]],
    )))
else:
    display(Markdown("Missing DeepPCB manifest. DeepPCB experiments are skipped in this notebook."))


## 8. Current Takeaways

From the saved fold-0 VisA PCB results:

- Multi-scale DINOv2 improves the heatmap baseline strongly on mean pixel AUROC, F1, and IoU.
- SAM2 refinement improves when prompted from multi-scale heatmaps, but the raw multi-scale heatmap threshold is still the strongest mask result so far.
- The next research step is prompt and mask-selection tuning: region thresholds, number of regions, point placement, area constraints, and failure-case inspection.

This notebook intentionally keeps the heavy computation in scripts and reusable code, while providing one readable place to inspect the whole project pipeline.
